<a href="https://colab.research.google.com/github/commandantekaustav/cocoon-shit/blob/main/scraper_roughs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install curl_cffi beautifulsoup4 pandas lxml

In [2]:
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

def fetch_sitemap_links(base_url):
    """
    Attempts to locate and parse XML sitemaps to extract all published article URLs.
    Utilizes curl_cffi to impersonate Chrome 120 TLS fingerprints.
    """
    # Common standard sitemap locations
    sitemap_endpoints = [
        "/sitemap_index.xml",
        "/post-sitemap.xml",
        "/sitemap.xml"
    ]

    valid_silk_links = []

    for endpoint in sitemap_endpoints:
        target_url = f"{base_url}{endpoint}"
        print(f"Testing sitemap endpoint: {target_url}")

        try:
            # impersonate="chrome120" bypasses WAF TLS fingerprinting blocks
            response = requests.get(target_url, impersonate="chrome120", timeout=15)

            if response.status_code == 200:
                print(f"Sitemap HTTP 200 OK. Parsing XML data...")

                # Check if it's a sitemap index containing sub-sitemaps
                soup = BeautifulSoup(response.content, "xml")
                sitemaps = soup.find_all("sitemap")

                # If sub-sitemaps exist, we need to extract from those
                urls_to_parse = []
                if sitemaps:
                    for sm in sitemaps:
                        urls_to_parse.append(sm.find("loc").text)
                else:
                    urls_to_parse.append(target_url)

                # Parse the actual post URLs
                for xml_url in urls_to_parse:
                    sub_response = requests.get(xml_url, impersonate="chrome120", timeout=15)
                    sub_soup = BeautifulSoup(sub_response.content, "xml")
                    locations = sub_soup.find_all("loc")

                    for loc in locations:
                        url = loc.text
                        # Filter for URLs containing our target keyword
                        if "silk-cocoon" in url.lower():
                            valid_silk_links.append(url)

                if valid_silk_links:
                    break # Successfully found and parsed the correct structure

        except Exception as e:
            print(f"Failed to fetch {target_url}: {str(e)}")

    return list(set(valid_silk_links))

# --- Execution ---
base_target = "https://kannadatopnews.com"
print("Initiating WAF-bypassing sitemap extraction...")

active_urls = fetch_sitemap_links(base_target)

print(f"\n--- Extraction Complete ---")
print(f"Successfully identified {len(active_urls)} active silk market links.")

if active_urls:
    os.makedirs("output", exist_ok=True)
    df = pd.DataFrame(active_urls, columns=["url"])
    df.to_csv("output/active_links_from_sitemap.csv", index=False)
    print("Links saved to output/active_links_from_sitemap.csv")

Initiating WAF-bypassing sitemap extraction...
Testing sitemap endpoint: https://kannadatopnews.com/sitemap_index.xml
Sitemap HTTP 200 OK. Parsing XML data...

--- Extraction Complete ---
Successfully identified 13925 active silk market links.
Links saved to output/active_links_from_sitemap.csv


In [3]:
import asyncio
import pandas as pd
from bs4 import BeautifulSoup
import re
from datetime import datetime
import nest_asyncio
from curl_cffi.requests import AsyncSession
import os
from io import StringIO

# Apply the Jupyter patch for asyncio
nest_asyncio.apply()

# --- 1. Load the Target Links ---
df_links = pd.read_csv('/content/output/active_links_from_sitemap.csv')
target_urls = df_links['url'].tolist()
total_urls = len(target_urls)

def clean_variety(variety_str):
    """Normalizes cross-breed and bivoltine variations."""
    variety_str = str(variety_str).lower()
    if 'cross' in variety_str or 'cb' in variety_str: return 'CB'
    if 'voltine' in variety_str or 'bv' in variety_str: return 'BV'
    return variety_str

def parse_html_table(html_content, url):
    """Extracts data and forces compliance with the 9-column target schema."""
    data_records = []
    try:
        soup = BeautifulSoup(html_content, 'lxml')
        page_text = soup.get_text()

        # Isolate Market Name
        market_match = re.search(r'([A-Za-z]+)\s+Government\s+Silk\s+Cocoon', page_text, re.IGNORECASE)
        market_name = market_match.group(1).upper() if market_match else "UNKNOWN"

        # Isolate and format Date
        date_match = re.search(r'Date:\s*([\d/]+)', page_text, re.IGNORECASE)
        if date_match:
            raw_date = date_match.group(1)
            date_str = datetime.strptime(raw_date, "%d/%m/%Y").strftime("%Y-%m-%d")
        else:
            date_str = "UNKNOWN"

        table_element = soup.find('table')
        if not table_element:
            return []

        df_list = pd.read_html(StringIO(str(table_element)))
        if not df_list:
            return []

        df = df_list[0]
        df.columns = [str(c).strip().lower() for c in df.columns]

        # Dynamic column identification logic to handle legacy site changes
        variety_col = [c for c in df.columns if 'variety' in c or 'ಗೂಡು' in c][0]
        lots_col = [c for c in df.columns if 'lots' in c][0]
        qty_col = [c for c in df.columns if 'qty' in c or 'kg' in c][0]
        min_col = [c for c in df.columns if 'min' in c][0]
        max_col = [c for c in df.columns if 'max' in c][0]
        avg_col = [c for c in df.columns if 'avg' in c][0]

        for _, row in df.iterrows():
            clean_min = re.sub(r'[^\d\.]', '', str(row[min_col]))
            clean_max = re.sub(r'[^\d\.]', '', str(row[max_col]))
            clean_avg = re.sub(r'[^\d\.]', '', str(row[avg_col]))
            clean_qty = re.sub(r'[^\d\.]', '', str(row[qty_col]))
            clean_lots = re.sub(r'[^\d]', '', str(row[lots_col]))

            data_records.append({
                "market": market_name,
                "date": date_str,
                "variety": clean_variety(row[variety_col]),
                "lots": int(clean_lots) if clean_lots else None,
                "qty_kg": float(clean_qty) if clean_qty else None,
                "min_price": float(clean_min) if clean_min else None,
                "max_price": float(clean_max) if clean_max else None,
                "avg_price": float(clean_avg) if clean_avg else None,
                "source_url": url
            })
    except Exception:
        # Silently fail isolated structural anomalies to preserve execution flow
        pass
    return data_records

async def fetch_and_parse(session, url, semaphore):
    """Executes the asynchronous HTTP GET request governed by a concurrency semaphore."""
    async with semaphore:
        try:
            response = await session.get(url, timeout=15)
            if response.status_code == 200:
                return parse_html_table(response.content, url)
        except Exception:
            return []
    return []

async def execute_scraping():
    # Concurrency limit set to 15. Scaling higher increases the probability of TLS handshake timeouts.
    CONCURRENCY_LIMIT = 15
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    csv_filename = "master_cocoon_prices.csv"
    columns = ['market', 'date', 'variety', 'lots', 'qty_kg', 'min_price', 'max_price', 'avg_price', 'source_url']

    # Initialize the empty master CSV with the required headers
    pd.DataFrame(columns=columns).to_csv(csv_filename, index=False)

    print(f"Initializing scraping architecture for {total_urls} validated endpoints...")
    total_valid_rows = 0

    async with AsyncSession(impersonate="chrome120") as session:
        batch_size = 500

        for i in range(0, total_urls, batch_size):
            batch = target_urls[i:i + batch_size]
            tasks = [fetch_and_parse(session, url, semaphore) for url in batch]
            results = await asyncio.gather(*tasks)

            batch_records = []
            for records in results:
                if records:
                    batch_records.extend(records)

            # Incremental disk writes eliminate memory leak risks
            if batch_records:
                batch_df = pd.DataFrame(batch_records)
                batch_df.to_csv(csv_filename, mode='a', header=False, index=False)
                total_valid_rows += len(batch_records)

            print(f"Processed {min(i + batch_size, total_urls)}/{total_urls} URLs. Extracted {total_valid_rows} tabular rows.", end="\r")

    print(f"\n\nExecution finalized. Stored {total_valid_rows} formatted data vectors inside '{csv_filename}'.")

# --- 3. Execute Execution Block ---
await execute_scraping()

Initializing scraping architecture for 13925 validated endpoints...
Processed 13925/13925 URLs. Extracted 640 tabular rows.

Execution finalized. Stored 640 formatted data vectors inside 'master_cocoon_prices.csv'.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import asyncio
import pandas as pd
from bs4 import BeautifulSoup
import re
from datetime import datetime
import nest_asyncio
from curl_cffi.requests import AsyncSession
import os
from io import StringIO
from tqdm.auto import tqdm

# Apply the Jupyter patch
nest_asyncio.apply()

# 1. Define Google Drive paths for persistent storage
BASE_DIR = '/content/drive/MyDrive/Silk_Scraper'
os.makedirs(BASE_DIR, exist_ok=True)

MASTER_CSV = os.path.join(BASE_DIR, "master_cocoon_prices.csv")
STATE_FILE = os.path.join(BASE_DIR, "processed_urls.txt")
EMPTY_LOG = os.path.join(BASE_DIR, "empty_urls.txt")

# Initialize the master CSV with headers if it doesn't exist yet
if not os.path.exists(MASTER_CSV):
    columns = ['market', 'date', 'variety', 'lots', 'qty_kg', 'min_price', 'max_price', 'avg_price', 'source_url']
    pd.DataFrame(columns=columns).to_csv(MASTER_CSV, index=False)

# Load the Target Links
df_links = pd.read_csv('/content/drive/MyDrive/Silk_Scraper/active_links_from_sitemap.csv')
all_target_urls = df_links['url'].tolist()

def clean_variety(variety_str):
    variety_str = str(variety_str).lower()
    if 'cross' in variety_str or 'cb' in variety_str: return 'CB'
    if 'voltine' in variety_str or 'bv' in variety_str: return 'BV'
    return variety_str

def parse_html_table(html_content, url):
    data_records = []
    try:
        soup = BeautifulSoup(html_content, 'lxml')
        page_text = soup.get_text()

        market_match = re.search(r'([A-Za-z]+)\s+Government\s+Silk\s+Cocoon', page_text, re.IGNORECASE)
        market_name = market_match.group(1).upper() if market_match else "UNKNOWN"

        date_match = re.search(r'Date:\s*([\d/]+)', page_text, re.IGNORECASE)
        date_str = datetime.strptime(date_match.group(1), "%d/%m/%Y").strftime("%Y-%m-%d") if date_match else "UNKNOWN"

        table_element = soup.find('table')
        if not table_element:
            return []

        # Using StringIO to fix the pandas Future Warning
        df_list = pd.read_html(StringIO(str(table_element)))
        if not df_list:
            return []

        df = df_list[0]
        df.columns = [str(c).strip().lower() for c in df.columns]

        variety_col = [c for c in df.columns if 'variety' in c or 'ಗೂಡು' in c][0]
        lots_col = [c for c in df.columns if 'lots' in c][0]
        qty_col = [c for c in df.columns if 'qty' in c or 'kg' in c][0]
        min_col = [c for c in df.columns if 'min' in c][0]
        max_col = [c for c in df.columns if 'max' in c][0]
        avg_col = [c for c in df.columns if 'avg' in c][0]

        for _, row in df.iterrows():
            clean_min = re.sub(r'[^\d\.]', '', str(row[min_col]))
            clean_max = re.sub(r'[^\d\.]', '', str(row[max_col]))
            clean_avg = re.sub(r'[^\d\.]', '', str(row[avg_col]))
            clean_qty = re.sub(r'[^\d\.]', '', str(row[qty_col]))
            clean_lots = re.sub(r'[^\d]', '', str(row[lots_col]))

            data_records.append({
                "market": market_name, "date": date_str, "variety": clean_variety(row[variety_col]),
                "lots": int(clean_lots) if clean_lots else None,
                "qty_kg": float(clean_qty) if clean_qty else None,
                "min_price": float(clean_min) if clean_min else None,
                "max_price": float(clean_max) if clean_max else None,
                "avg_price": float(clean_avg) if clean_avg else None,
                "source_url": url
            })
    except Exception:
        pass
    return data_records

async def fetch_and_parse(session, url, semaphore):
    async with semaphore:
        try:
            response = await session.get(url, timeout=15)
            if response.status_code == 200:
                records = parse_html_table(response.content, url)
                return url, records
        except Exception:
            pass
        return url, []

async def execute_scraping():
    # Load state to see what we've already done
    processed_urls = set()
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, 'r') as f:
            processed_urls = set(line.strip() for line in f)

    # Only process URLs that are NOT in the processed list
    remaining_urls = [url for url in all_target_urls if url not in processed_urls]
    total_remaining = len(remaining_urls)

    if total_remaining == 0:
        print("All URLs have already been processed.")
        return

    print(f"Resuming execution. {len(processed_urls)} already completed. {total_remaining} left to scan.")

    CONCURRENCY_LIMIT = 15
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    async with AsyncSession(impersonate="chrome120") as session:
        batch_size = 500

        # Initialize tqdm progress bar
        with tqdm(total=total_remaining, desc="Scraping Progress") as pbar:
            for i in range(0, total_remaining, batch_size):
                batch = remaining_urls[i:i + batch_size]
                tasks = [fetch_and_parse(session, url, semaphore) for url in batch]
                results = await asyncio.gather(*tasks)

                batch_records = []
                newly_processed = []
                empty_urls = []

                for url, records in results:
                    newly_processed.append(url)
                    if records:
                        batch_records.extend(records)
                    else:
                        empty_urls.append(url)

                # 1. Append valid data to master CSV
                if batch_records:
                    pd.DataFrame(batch_records).to_csv(MASTER_CSV, mode='a', header=False, index=False)

                # 2. Append empty URLs to the fail log for your review
                if empty_urls:
                    with open(EMPTY_LOG, 'a') as f:
                        for url in empty_urls:
                            f.write(url + "\n")

                # 3. Append successfully scanned URLs to state file so they aren't repeated on crash
                with open(STATE_FILE, 'a') as f:
                    for url in newly_processed:
                        f.write(url + "\n")

                pbar.update(len(batch))

# Execute the asynchronous block
await execute_scraping()

Resuming execution. 0 already completed. 13925 left to scan.


Scraping Progress:   0%|          | 0/13925 [00:00<?, ?it/s]